In [7]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [8]:
df = pd.read_csv('quote_dataset.csv')

In [9]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [10]:
quotes = quotes.str.lower()
import string
translator = str.maketrans('','',string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer
vocab_size = 8978
tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(quotes)
word_index = tokenizer.word_index
print(len(word_index))
list(word_index.items())[::10]

8978


[('the', 1),
 ('in', 11),
 ('what', 21),
 ('do', 31),
 ('people', 41),
 ('just', 51),
 ('things', 61),
 ('an', 71),
 ('live', 81),
 ('thing', 91),
 ('other', 101),
 ('day', 111),
 ('been', 121),
 ('feel', 131),
 ('back', 141),
 ('great', 151),
 ('true', 161),
 ('else', 171),
 ('matter', 181),
 ('ive', 191),
 ('two', 201),
 ('pain', 211),
 ('here', 221),
 ('forever', 231),
 ('forget', 241),
 ('future', 251),
 ('old', 261),
 ('stay', 271),
 ('work', 281),
 ('story', 291),
 ('sad', 301),
 ('against', 311),
 ('told', 321),
 ('worth', 331),
 ('past', 341),
 ('yet', 351),
 ('lost', 361),
 ('ask', 371),
 ('such', 381),
 ('mistakes', 391),
 ('deep', 401),
 ('knew', 411),
 ('darkness', 421),
 ('whats', 431),
 ('nobody', 441),
 ('dare', 451),
 ('fight', 461),
 ('turn', 471),
 ('mother', 481),
 ('took', 491),
 ('learned', 501),
 ('gets', 511),
 ('finally', 521),
 ('teach', 531),
 ('came', 541),
 ('front', 551),
 ('longer', 561),
 ('blood', 571),
 ('brave', 581),
 ('powerful', 591),
 ('pleasure', 

In [12]:
sequence = tokenizer.texts_to_sequences(quotes)

In [13]:
x=[]
y=[]
for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        x.append(input_seq)
        y.append(output_seq)

In [14]:
max_len = max(len(x) for x in x)
print(max_len)

745


In [15]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(x,maxlen=max_len, padding='pre')

In [16]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [17]:
y = np.array(y)

In [18]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y,num_classes=vocab_size)
y_one_hot.shape

(85270, 8978)

In [19]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM, SimpleRNN

In [20]:
embeddings_dim = 100
rnn_units = 128

In [21]:
rnn_model = Sequential()
rnn_model.add(Embedding(input_dim=vocab_size, output_dim=embeddings_dim, input_length=max_len))
rnn_model.add(SimpleRNN(units = rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

In [22]:
rnn_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [23]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [24]:
lstm_model = Sequential()
lstm_model.add(Embedding(input_dim=vocab_size, output_dim=embeddings_dim, input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [25]:
epochs = 10
batch_size = 128

In [ ]:
history_rnn = rnn_model.fit(X_padded,y_one_hot,epochs=epochs,batch_size = batch_size, validation_split=0.1)

In [26]:
lstm_model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

In [27]:
history_lstm = lstm_model.fit(X_padded,y_one_hot,epochs=100,batch_size = batch_size, validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 55ms/step - accuracy: 0.0341 - loss: 7.0780 - val_accuracy: 0.0503 - val_loss: 6.6280
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.0603 - loss: 6.2793 - val_accuracy: 0.0779 - val_loss: 6.4485
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 56ms/step - accuracy: 0.0871 - loss: 5.9652 - val_accuracy: 0.0965 - val_loss: 6.3680
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 34s 57ms/step - accuracy: 0.1102 - loss: 5.6745 - val_accuracy: 0.1032 - val_loss: 6.3288
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - accuracy: 0.1230 - loss: 5.4522 - val_accuracy: 0.1086 - val_loss: 6.3115
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 58ms/step - accuracy: 0.1331 - loss: 5.2698 - val_accuracy: 0.1095 - val_loss: 6.3454
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - accuracy: 0.1460 - loss: 5.0859 - val_accuracy: 0.1129 - val_loss: 6.3906
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 35s 58ms/step - accuracy: 0.1554 - loss: 4

In [28]:
lstm_model.save('lstm_model.h5')

In [29]:
index_to_word = {}
for word, index in word_index.items():
    index_to_word[index] = word

In [30]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [32]:

def predictor(model,tokenizer,text,max_len):
  text = text.lower()
  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen=max_len,padding='pre')
  pred = model.predict(seq)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [33]:
seed_text = 'life is'
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 258ms/step
hard


In [52]:
seed_text = 'the work was'
next_word = predictor(lstm_model,tokenizer,seed_text,max_len)
print(next_word)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
to


In [53]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [54]:
seed = "are you a "
generated_text = generate_text(lstm_model,tokenizer,seed,max_len,10)
print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
are you a  other thing than than a book a life is that


In [57]:
seed = "your soul is the best when "
generated_text = generate_text(lstm_model,tokenizer,seed,max_len,20)
print(generated_text)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step
your soul is the best when  you can be disappointed with what your heart is to be around and when she is left and nothing is


In [59]:
import pickle
with open('tokenizer.pkl','wb') as f:
    pickle.dump(tokenizer,f)

In [60]:
with open('max_len.pkl','wb') as f:
    pickle.dump(max_len,f)